In [1]:
# --- Importación de librerias y configuración ---
import pandas as pd
import numpy as np
import os

# Ruta a los datos crudos
RAW_PATH = '../data/raw/'

# Para verificar que la carpeta existe
print("Archivos en data/raw/:")
for f in os.listdir(RAW_PATH):
    print(f"  {f}")

Archivos en data/raw/:
  .gitkeep
  .ipynb_checkpoints
  DOWNLOAD_INSTRUCTIONS.md
  employee_survey_data.csv
  general_data.csv
  glassdoor_reviews.csv
  in_time.csv
  manager_survey_data.csv
  out_time.csv
  reference
  WA_Fn-UseC_-HR-Employee-Attrition.csv


In [2]:
# --- Carga IBM HR (dataset principal) ---
ibm = pd.read_csv(RAW_PATH + 'WA_Fn-UseC_-HR-Employee-Attrition.csv')

print(f"Shape: {ibm.shape}")
print(f"\nTipos de datos:\n{ibm.dtypes}")
print(f"\nPrimeras 3 filas:")
ibm.head(3)

Shape: (1470, 35)

Tipos de datos:
Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingY

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


In [3]:
# --- Exploración básica dataset IBM HR ---
print("=== VALORES NULOS ===")
print(ibm.isnull().sum().sum(), "nulos en total")

print("\n=== BALANCE DEL TARGET ===")
print(ibm['Attrition'].value_counts(normalize=True).round(3))

print("\n=== VARIABLES CONSTANTES (un solo valor único) ===")
constantes = ibm.columns[ibm.nunique() == 1].tolist()
print(constantes)

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
ibm.describe().round(2)

=== VALORES NULOS ===
0 nulos en total

=== BALANCE DEL TARGET ===
Attrition
No     0.839
Yes    0.161
Name: proportion, dtype: float64

=== VARIABLES CONSTANTES (un solo valor único) ===
['EmployeeCount', 'Over18', 'StandardHours']

=== ESTADÍSTICAS DESCRIPTIVAS ===


,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
count,1470.00,1470.00,1470.00,1470.00,1470.0,1470.00,1470.00,1470.00,1470.00,1470.00,...,1470.00,1470.0,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00
mean,36.92,802.49,9.19,2.91,1.0,1024.87,2.72,65.89,2.73,2.06,...,2.71,80.0,0.79,11.28,2.80,2.76,7.01,4.23,2.19,4.12
std,9.14,403.51,8.11,1.02,0.0,602.02,1.09,20.33,0.71,1.11,...,1.08,0.0,0.85,7.78,1.29,0.71,6.13,3.62,3.22,3.57
min,18.00,102.00,1.00,1.00,1.0,1.00,1.00,30.00,1.00,1.00,...,1.00,80.0,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
25%,30.00,465.00,2.00,2.00,1.0,491.25,2.00,48.00,2.00,1.00,...,2.00,80.0,0.00,6.00,2.00,2.00,3.00,2.00,0.00,2.00
50%,36.00,802.00,7.00,3.00,1.0,1020.50,3.00,66.00,3.00,2.00,...,3.00,80.0,1.00,10.00,3.00,3.00,5.00,3.00,1.00,3.00
75%,43.00,1157.00,14.00,4.00,1.0,1555.75,4.00,83.75,3.00,3.00,...,4.00,80.0,1.00,15.00,3.00,3.00,9.00,7.00,3.00,7.00
max,60.00,1499.00,29.00,5.00,1.0,2068.00,4.00,100.00,4.00,5.00,...,4.00,80.0,3.00,40.00,6.00,4.00,40.00,18.00,15.00,17.00


In [4]:
# --- Carga dataset Extended (para validación) ---
# Este dataset tiene varios CSV que se unen por EmployeeID como primary key
general = pd.read_csv(RAW_PATH + 'general_data.csv')
survey = pd.read_csv(RAW_PATH + 'employee_survey_data.csv')
manager = pd.read_csv(RAW_PATH + 'manager_survey_data.csv')

print(f"general_data: {general.shape}")
print(f"employee_survey: {survey.shape}")
print(f"manager_survey: {manager.shape}")

# Unir por EmployeeID
extended = general.merge(survey, on='EmployeeID').merge(manager, on='EmployeeID')
print(f"\nDataset Extended unido: {extended.shape}")
print(f"Nulos totales: {extended.isnull().sum().sum()}")
print(f"\nBalance del target:")
print(extended['Attrition'].value_counts(normalize=True).round(3))


general_data: (4410, 24)
employee_survey: (4410, 4)
manager_survey: (4410, 3)

Dataset Extended unido: (4410, 29)
Nulos totales: 111

Balance del target:
Attrition
No     0.839
Yes    0.161
Name: proportion, dtype: float64


In [5]:
# --- Carga Glassdoor Reviews (para NLP) ---
glassdoor = pd.read_csv(RAW_PATH + 'glassdoor_reviews.csv')

print(f"Shape: {glassdoor.shape}")
print(f"\nColumnas:\n{glassdoor.columns.tolist()}")
print(f"\nNulos por columna:")
print(glassdoor.isnull().sum())
print(f"\nEjemplo de una reseña:")
print(glassdoor.iloc[0])

Shape: (838566, 18)

Columnas:
['firm', 'date_review', 'job_title', 'current', 'location', 'overall_rating', 'work_life_balance', 'culture_values', 'diversity_inclusion', 'career_opp', 'comp_benefits', 'senior_mgmt', 'recommend', 'ceo_approv', 'outlook', 'headline', 'pros', 'cons']

Nulos por columna:
firm                        0
date_review                 0
job_title                   0
current                     0
location               297343
overall_rating              0
work_life_balance      149894
culture_values         191373
diversity_inclusion    702500
career_opp             147501
comp_benefits          150082
senior_mgmt            155876
recommend                   0
ceo_approv                  0
outlook                     0
headline                 2590
pros                        2
cons                       13
dtype: int64

Ejemplo de una reseña:
firm                                               AFH-Wealth-Management
date_review                                    

In [6]:
# --- Clasificación tipos de variable del dataset IBM HR ---
# Definido manualmente el tipo de cada variable
tipo_variable = {
    # Numéricas continuas
    'Age': 'Numérica continua',
    'DailyRate': 'Numérica continua',
    'DistanceFromHome': 'Numérica continua',
    'HourlyRate': 'Numérica continua',
    'MonthlyIncome': 'Numérica continua',
    'MonthlyRate': 'Numérica continua',
    'NumCompaniesWorked': 'Numérica discreta',
    'PercentSalaryHike': 'Numérica continua',
    'TotalWorkingYears': 'Numérica discreta',
    'TrainingTimesLastYear': 'Numérica discreta',
    'YearsAtCompany': 'Numérica discreta',
    'YearsInCurrentRole': 'Numérica discreta',
    'YearsSinceLastPromotion': 'Numérica discreta',
    'YearsWithCurrManager': 'Numérica discreta',
    # Ordinales (escala 1-4 o 1-5)
    'Education': 'Ordinal (1-5)',
    'EnvironmentSatisfaction': 'Ordinal (1-4)',
    'JobInvolvement': 'Ordinal (1-4)',
    'JobLevel': 'Ordinal (1-5)',
    'JobSatisfaction': 'Ordinal (1-4)',
    'PerformanceRating': 'Ordinal (1-4)',
    'RelationshipSatisfaction': 'Ordinal (1-4)',
    'StockOptionLevel': 'Ordinal (0-3)',
    'WorkLifeBalance': 'Ordinal (1-4)',
    # Categóricas nominales
    'Attrition': 'Target (Yes/No)',
    'BusinessTravel': 'Categórica nominal',
    'Department': 'Categórica nominal',
    'EducationField': 'Categórica nominal',
    'Gender': 'Categórica nominal',
    'JobRole': 'Categórica nominal',
    'MaritalStatus': 'Categórica nominal',
    'OverTime': 'Categórica nominal',
    # Constantes / ID (a eliminar en EDA)
    'EmployeeCount': 'CONSTANTE — eliminar',
    'EmployeeNumber': 'ID — eliminar',
    'Over18': 'CONSTANTE — eliminar',
    'StandardHours': 'CONSTANTE — eliminar',
}

# Verificación de que se cubren todas las columnas
faltantes = set(ibm.columns) - set(tipo_variable.keys())
if faltantes:
    print(f" Variables sin clasificar: {faltantes}")
else:
    print(" Todas las variables clasificadas")

 Todas las variables clasificadas


In [7]:
!pip install openpyxl

In [8]:
# --- Generación de diccionario de variables ---
diccionario = pd.DataFrame({
    'Variable': ibm.columns,
    'Tipo_Python': ibm.dtypes.values,
    'Tipo_Analisis': [tipo_variable.get(col, 'Sin clasificar') for col in ibm.columns],
    'Valores_Unicos': ibm.nunique().values,
    'Nulos': ibm.isnull().sum().values,
    'Min': [ibm[col].min() if ibm[col].dtype != 'object' else ibm[col].unique()[0] for col in ibm.columns],
    'Max': [ibm[col].max() if ibm[col].dtype != 'object' else ibm[col].unique()[-1] for col in ibm.columns],
    'Notas_RRHH': ''  # Columna completada por Kalil
})

# Exportar a Excel
output_path = '../reports/Diccionario_Variables_IBM_HR.xlsx'
diccionario.to_excel(output_path, index=False, sheet_name='IBM_HR')
print(f" Diccionario exportado a: {output_path}")
diccionario

 Diccionario exportado a: ../reports/Diccionario_Variables_IBM_HR.xlsx


,Variable,Tipo_Python,Tipo_Analisis,Valores_Unicos,Nulos,Min,Max,Notas_RRHH
0,Age,int64,Numérica continua,43,0,18,60,
1,Attrition,str,Target (Yes/No),2,0,No,Yes,
2,BusinessTravel,str,Categórica nominal,3,0,Non-Travel,Travel_Rarely,
3,DailyRate,int64,Numérica continua,886,0,102,1499,
4,Department,str,Categórica nominal,3,0,Human Resources,Sales,
5,DistanceFromHome,int64,Numérica continua,29,0,1,29,
6,Education,int64,Ordinal (1-5),5,0,1,5,
7,EducationField,str,Categórica nominal,6,0,Human Resources,Technical Degree,
8,EmployeeCount,int64,CONSTANTE — eliminar,1,0,1,1,
9,EmployeeNumber,int64,ID — eliminar,1470,0,1,2068,


In [9]:
# --- Resumen comparativo de los 3 datasets ---
resumen = pd.DataFrame({
    'Dataset': ['IBM HR (principal)', 'Extended (validación)', 'Glassdoor (NLP)'],
    'Filas': [ibm.shape[0], extended.shape[0], glassdoor.shape[0]],
    'Columnas': [ibm.shape[1], extended.shape[1], glassdoor.shape[1]],
    'Nulos_Totales': [ibm.isnull().sum().sum(), extended.isnull().sum().sum(), glassdoor.isnull().sum().sum()],
    'Tiene_Target': ['Sí (Attrition)', 'Sí (Attrition)', 'No (sentimiento implícito)'],
    'Uso': ['Entrenamiento 5 modelos ML', 'Validación externa', 'Pipeline NLP sentimiento']
})

print("=== RESUMEN DE DATASETS DEL TFM ===\n")
resumen

=== RESUMEN DE DATASETS DEL TFM ===



,Dataset,Filas,Columnas,Nulos_Totales,Tiene_Target,Uso
0,IBM HR (principal),1470,35,0,Sí (Attrition),Entrenamiento 5 modelos ML
1,Extended (validación),4410,29,111,Sí (Attrition),Validación externa
2,Glassdoor (NLP),838566,18,1797174,No (sentimiento implícito),Pipeline NLP sentimiento
